In [15]:
from substack_api import Newsletter
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from datetime import datetime
from pathlib import Path
import re
import os

In [16]:
def safe_get(obj, attr, default=None):
    if obj is None:
        return default

    # method: obj.attr()
    method = getattr(obj, attr, None)
    if callable(method):
        try:
            value = method()
            if value is not None:
                return value
        except Exception:
            pass

    # attribute: obj.attr
    try:
        value = getattr(obj, attr, None)
        if value is not None:
            return value
    except Exception:
        pass

    # dict: obj[attr]
    if isinstance(obj, dict):
        value = obj.get(attr, default)
        if value is not None:
            return value

    return default

In [17]:
def object_to_dict(obj):
    """
    Best-effort conversion of library objects into dicts.
    """
    if obj is None:
        return {}

    if isinstance(obj, dict):
        return obj

    # Try common object storage
    try:
        if hasattr(obj, "__dict__"):
            return {
                k: v for k, v in vars(obj).items()
                if not k.startswith("_")
            }
    except Exception:
        pass

    return {}

In [18]:
def deep_get(data, keys, default=""):
    """
    Look for the first matching key in flat or nested dict/object structures.
    """
    if not data:
        return default

    for key in keys:
        # direct dict lookup
        if isinstance(data, dict) and key in data and data[key] is not None:
            return data[key]

        # attribute lookup
        try:
            value = getattr(data, key, None)
            if value is not None:
                return value
        except Exception:
            pass

    # search one level deeper
    if isinstance(data, dict):
        for _, value in data.items():
            if isinstance(value, dict):
                result = deep_get(value, keys, default=None)
                if result is not None:
                    return result
            else:
                try:
                    inner = object_to_dict(value)
                    if inner:
                        result = deep_get(inner, keys, default=None)
                        if result is not None:
                            return result
                except Exception:
                    pass

    return default

In [19]:
def clean_html_to_markdown(html: str) -> str:
    soup = BeautifulSoup(html or "", "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    clean_html = str(soup)
    markdown = md(clean_html, heading_style="ATX")

    # cleanup
    markdown = re.sub(r"\n{3,}", "\n\n", markdown)
    markdown = re.sub(r"^\#\s+\*\*(.*?)\*\*$", r"# \1", markdown, flags=re.MULTILINE)
    markdown = re.sub(r"^\#\#\s+\*\*(.*?)\*\*$", r"## \1", markdown, flags=re.MULTILINE)
    markdown = markdown.strip()

    return markdown

In [20]:
def format_date(value) -> str:
    if value is None or value == "":
        return ""

    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d")

    text = str(value).strip()

    # unix timestamp
    if text.isdigit():
        try:
            return datetime.fromtimestamp(int(text)).strftime("%Y-%m-%d")
        except Exception:
            pass

    # ISO-like string
    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00")).strftime("%Y-%m-%d")
    except Exception:
        return text

In [21]:
def yaml_escape(value) -> str:
    if value is None:
        return '""'
    text = str(value).replace('"', '\\"')
    return f'"{text}"'


def sanitize_filename(name: str, fallback: str = "post") -> str:
    text = (name or "").strip()
    if not text:
        text = fallback
    text = re.sub(r'[<>:"/\\|?*]+', "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:150]

In [22]:
def extract_metadata_from_html(html: str) -> dict:
    soup = BeautifulSoup(html or "", "html.parser")

    def get_meta(*names):
        for name in names:
            tag = soup.find("meta", attrs={"name": name}) or soup.find("meta", attrs={"property": name})
            if tag and tag.get("content"):
                return tag["content"].strip()
        return ""

    canonical_url = ""
    canonical_tag = soup.find("link", rel="canonical")
    if canonical_tag and canonical_tag.get("href"):
        canonical_url = canonical_tag["href"].strip()

    title = ""
    if soup.title and soup.title.string:
        title = soup.title.string.strip()

    h1 = soup.find("h1")
    if not title and h1:
        title = h1.get_text(" ", strip=True)

    post_date = ""
    time_tag = soup.find("time")
    if time_tag:
        post_date = time_tag.get("datetime", "").strip() or time_tag.get_text(" ", strip=True)

    return {
        "canonical_url": canonical_url,
        "title": title,
        "description": get_meta("description", "og:description", "twitter:description"),
        "post_date": post_date,
    }

In [ ]:
def extract_post_record(post) -> dict:
    raw = object_to_dict(post)

    body_html = (
        safe_get(post, "body_html", "")
        or safe_get(post, "content_html", "")
        or safe_get(post, "get_content", "")
        or deep_get(raw, ["body_html", "content_html", "html", "body"], "")
    )

    html_meta = extract_metadata_from_html(body_html)

    title = (
        safe_get(post, "title", "")
        or deep_get(raw, ["title", "headline", "name"], "")
        or html_meta["title"]
    )

    description = (
        safe_get(post, "description", "")
        or safe_get(post, "subtitle", "")
        or deep_get(raw, ["description", "subtitle", "excerpt", "summary"], "")
        or html_meta["description"]
    )

    canonical_url = (
        safe_get(post, "canonical_url", "")
        or safe_get(post, "url", "")
        or deep_get(raw, ["canonical_url", "url", "canonicalUrl"], "")
        or html_meta["canonical_url"]
    )

    slug = (
        safe_get(post, "slug", "")
        or deep_get(raw, ["slug"], "")
    )

    post_date = (
        safe_get(post, "post_date", "")
    )

    body_markdown = clean_html_to_markdown(body_html)

    return {
        "canonical_url": canonical_url or "",
        "slug": slug or "",
        "post_date": format_date(post_date),
        "title": title or "",
        "description": description or "",
        "body_html": body_html or "",
        "body_markdown": body_markdown or "",
    }

In [24]:
def post_to_markdown(posts, output_path: str):
    if posts is None:
        return

    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not isinstance(posts, list):
        posts = list(posts)

    for idx, post in enumerate(posts, start=1):
        record = extract_post_record(post)

        filename_base = record["slug"] or record["title"] or f"post_{idx}"
        filename = sanitize_filename(filename_base, fallback=f"post_{idx}") + ".md"

        output = f"""---
title: {yaml_escape(record["title"])}
description: {yaml_escape(record["description"])}
date: {yaml_escape(record["post_date"])}
type: "newsletter"
canonical_url: {yaml_escape(record["canonical_url"])}
slug: {yaml_escape(record["slug"])}
---

{record["body_markdown"]}
"""

        with open(output_dir / filename, "w", encoding="utf-8") as f:
            f.write(output)

        print(f"Saved: {output_dir / filename}")


def process_newsletter(base_url: str, limit: int):
    newsletter = Newsletter(base_url)
    posts = newsletter.get_posts(limit=limit)
    post_to_markdown(posts, output_path="data/processed")

In [25]:
process_newsletter("https://warikoo.substack.com/", limit=1)

Saved: data/processed/203-so-this-is-how-it-must-feel.md


In [27]:
newsletter = Newsletter("https://warikoo.substack.com/")
posts = newsletter.get_posts(limit=1)

In [28]:
posts

[Post(url=https://warikoo.substack.com/p/203-so-this-is-how-it-must-feel)]

In [30]:
extract_post_record(posts[0])

{'canonical_url': 'https://warikoo.substack.com/p/203-so-this-is-how-it-must-feel',
 'slug': '203-so-this-is-how-it-must-feel',
 'post_date': '',
 'title': 'Loss of control is very unsettling',
 'description': '',
 'body_html': '<h1><strong>Loss of control is very unsettling</strong></h1><p>We are in Egypt this week, visiting the ruins of a civilization nearly 4,000 years old. <br>As a family, we love history - and Egypt is history on steroids, haha!</p><p>As a country, this place reminds me of India. <br>People are still largely poor. <br>There is a lot of construction all around. <br>The society is vastly different from western civilization - the culture goes back centuries, and the ways are very unique to the place. </p><p>And, much like India, it is a culture of constant negotiation. <br>EVERYTHING is up for negotiation. <br>The price of the hotel, the tour guide fee, the price of a memento, and even how much tip to give. <br>Everyone is constantly selling you something. <br>On the

In [31]:
posts[0].get_metadata() 

{'audience': 'everyone',
 'audience_before_archived': None,
 'canonical_url': 'https://warikoo.substack.com/p/203-so-this-is-how-it-must-feel',
 'default_comment_sort': None,
 'editor_v2': False,
 'exempt_from_archive_paywall': False,
 'free_unlock_required': False,
 'id': 191449497,
 'podcast_art_url': None,
 'podcast_duration': None,
 'podcast_preview_upload_id': None,
 'podcast_upload_id': None,
 'podcast_url': None,
 'post_date': '2026-03-20T03:30:52.940Z',
 'updated_at': '2026-03-20T03:32:23.735Z',
 'publication_id': 5212746,
 'search_engine_description': None,
 'search_engine_title': 'So this is how it must feel',
 'section_id': None,
 'should_send_free_preview': False,
 'show_guest_bios': True,
 'slug': '203-so-this-is-how-it-must-feel',
 'social_title': 'So this is how it must feel',
 'subtitle': "Life isn't about controlling the outcome.",
 'teaser_post_eligible': True,
 'title': 'So this is how it must feel',
 'type': 'newsletter',
 'video_upload_id': None,
 'write_comment_pe

In [33]:
safe_get(posts[0].get_metadata(), "post_date", "")
safe_get(posts[0].get_metadata(), "description", "")

'Loss of control is very unsettling'